# Combined Disturbance Terrain Evaluation Analysis

This notebook analyzes episode-based quadruped locomotion evaluation logs collected while robots traverse terrain under combined disturbances: payloads, shifted CoM, lateral pushes, and random moments about the CoM.

It keeps the previous scalar metric pipeline and adds:

- episode success/failure rate plots by method,
- completion-time summaries for successful terrain completions,
- heading error relative to desired heading `0.0` / forward direction `[1, 0, 0]`,
- event-triggered lateral drift around detected `rand_push` changes,
- compact rollout CSVs for plotting selected episodes over time.

The pipeline is memory-conscious: each raw CSV is loaded, analyzed, converted to compact tables, then removed from memory before the next file is loaded. A process-parallel runner is also included.

In [ ]:
import os, re, gc, ast, math
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
except ImportError:
    sns = None

## Configuration
Update these paths and lists to match your experiment directory and naming conventions.

`APPROACH_ORDER` can be set to either raw task names such as `"go1_pact"` or display labels such as `"PACT"`. `HIGHLIGHT_APPROACH` uses the same convention.

In [ ]:
EXP_FOLDER = Path("exp_data_corl_07/terrain_04")
RESULTS_DIR = EXP_FOLDER / "terrain_disturbance_analysis_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

APPROACHES = ["go1_pact", "go1_pos", "go1_tau", "go1_abl1", "go1_abl2", "go1_abl3"]
TERRAINS = ["plane", "rough", "slope_up", "slope_down", "stairs_up", "stairs_down", "discrete", "wave"]

TERRAIN_FILTER = None
APPROACH_FILTER = None
DISTURBANCE_FILTER = None

CONTROL_DT = 0.01
BASE_HEIGHT_TARGET = 0.30
EPS = 1e-8

# Lateral drift settings.
IGNORE_TIME = 0.10
RECOVERY_TIME = 0.75
PUSH_CHANGE_TOL = 1e-6
MIN_PUSH_MAG = 1e-6

JOINT_LIMITS = None
JOINT_TORQUE_LIMITS = None

# Preferred plot ordering can use either raw approach ids (e.g., "go1_pact")
# or display labels (e.g., "PACT"). Set to None to use dataframe order.
APPROACH_ORDER = ['go1_pos', 'go1_tau', 'go1_abl1', 'go1_abl2', 'go1_abl3', 'go1_pact']

# Approach to visually emphasize in plots. Can be raw id or display label.
HIGHLIGHT_APPROACH = "PC"
APPROACH_DISPLAY_NAMES = {
    "go1_pact": "PC",
    "go1_pos": "DW",
    "go1_tau": "TO",
    "go1_abl1": "A1",
    "go1_abl2": "A2",
    "go1_abl3": "A3",
    "go1_rl2ac": "RA",
}
RECOMPUTE_METRICS = True
MAX_WORKERS = 8
SAVE_EVERY = 5

EXPORT_ROLLOUTS = True
ROLLOUT_EPISODES_PER_FILE = 1
ROLLOUT_MAX_ROWS_PER_EPISODE = None


FIGURE_SIZE = (8.0, 3.6)
SAVE_AS_PDF = True

## Loading and filename parsing

In [ ]:
ARRAY_COLUMNS = [
    "base_cmd", "base_pose", "base_rpy", "dof_pose", "base_lin_vel", "base_ang_vel",
    "dof_vel", "proj_grav", "feet_pos", "tau_act", "grf", "q_des", "tau_ff", "tau_pd",
    "payload", "com_shift", "rand_push", "rand_wrench",
]


def string_to_array(value):
    if isinstance(value, str):
        try:
            return ast.literal_eval(value)
        except (ValueError, SyntaxError):
            return value
    return value


def get_csv_converters(columns=ARRAY_COLUMNS):
    return {c: string_to_array for c in columns}


def load_eval_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, converters=get_csv_converters())


def _display_approach_name(name):
    return APPROACH_DISPLAY_NAMES.get(str(name), str(name))


def parse_eval_filename(path: Path, approaches=APPROACHES, terrains=TERRAINS):
    stem = Path(path).stem
    if stem.endswith("_episodes"):
        stem = stem[:-len("_episodes")]

    approach = None
    rest = stem
    for a in sorted(approaches, key=len, reverse=True):
        prefix = a + "_"
        if stem == a or stem.startswith(prefix):
            approach = a
            rest = stem[len(prefix):]
            break

    terrain = None
    disturbance = rest
    for t in sorted(terrains, key=len, reverse=True):
        prefix = t + "_"
        if rest == t or rest.startswith(prefix):
            terrain = t
            disturbance = rest[len(prefix):]
            break

    if approach is None:
        toks = stem.split("_")
        approach = "_".join(toks[:2]) if len(toks) >= 2 else stem
    if terrain is None:
        terrain = "unknown"
    return approach, terrain, disturbance


def find_eval_files(exp_folder: Path, approaches=APPROACHES, terrains=TERRAINS):
    rows = []
    for path in sorted(exp_folder.rglob("*_episodes.csv")):
        approach, terrain, disturbance = parse_eval_filename(path, approaches, terrains)
        if APPROACH_FILTER is not None and approach not in APPROACH_FILTER:
            continue
        if TERRAIN_FILTER is not None and terrain not in TERRAIN_FILTER:
            continue
        if DISTURBANCE_FILTER is not None:
            filters = DISTURBANCE_FILTER if isinstance(DISTURBANCE_FILTER, (list, tuple, set)) else [DISTURBANCE_FILTER]
            if not any(str(f) in disturbance for f in filters):
                continue
        rows.append({
            "path": str(path),
            "approach": approach,
            "terrain": terrain,
            "disturbance_condition": disturbance,
            "approach_label": _display_approach_name(approach),
        })
    return pd.DataFrame(rows)

In [ ]:
file_table = find_eval_files(EXP_FOLDER)
print(f"Found {len(file_table)} files")
file_table.head(20)

## Existing metric utilities

In [ ]:
def as_array(df, column, valid_mask=None):
    if column not in df.columns:
        return None
    s = df[column]
    if valid_mask is not None:
        s = s.loc[valid_mask]
    if len(s) == 0:
        return None
    try:
        return np.asarray(s.to_list(), dtype=float)
    except Exception:
        try:
            return np.asarray(s.to_list())
        except Exception:
            return None


def safe_mean(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    return float(np.nan) if x.size == 0 else float(np.nanmean(x))


def safe_std(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    return float(np.nan) if x.size == 0 else float(np.nanstd(x))


def rmse(x):
    x = np.asarray(x, dtype=float)
    return float(np.sqrt(np.nanmean(np.square(x))))


def mae(x):
    x = np.asarray(x, dtype=float)
    return float(np.nanmean(np.abs(x)))


def mean_norm(x, axis=1, ord=1):
    return float(np.nanmean(np.linalg.norm(np.asarray(x, dtype=float), axis=axis, ord=ord)))


def std_norm(x, axis=1, ord=1):
    return float(np.nanstd(np.linalg.norm(np.asarray(x, dtype=float), axis=axis, ord=ord)))


def get_row_mask_for_metrics(df):
    if "valid_episode" in df.columns:
        return df["valid_episode"].astype(float).to_numpy() == 1
    return np.ones(len(df), dtype=bool)


def wrap_to_pi(x):
    return (np.asarray(x) + np.pi) % (2.0 * np.pi) - np.pi

In [ ]:
def compute_eval_metrics(
    df,
    approach="pact",
    base_height_target=0.30,
    joint_limits=None,
    joint_torque_limits=None,
    pact_metric_approaches=("pact", "abl", "rl2ac"),
    eps=1e-8,
):
    metrics = {}
    use_pact_metrics = approach is not None and any(k.lower() in approach.lower() for k in pact_metric_approaches)

    n_total = len(df)
    if "failure_reset" in df.columns:
        failure = df["failure_reset"].astype(float).to_numpy()
    elif "failure" in df.columns:
        failure = df["failure"].astype(float).to_numpy()
    else:
        failure = np.zeros(n_total)
    valid_mask = failure == 0

    metrics["num_rows_total"] = int(n_total)
    metrics["num_rows_valid"] = int(valid_mask.sum())
    metrics["num_failures_rowwise"] = int(failure.sum())
    metrics["failure_rate_rowwise"] = float(failure.mean()) if n_total else np.nan
    if valid_mask.sum() == 0:
        return metrics

    q_actions = as_array(df, "q_des", valid_mask)
    q_obs = as_array(df, "dof_pose", valid_mask)
    vel_cmds = as_array(df, "base_cmd", valid_mask)
    lin_vel = as_array(df, "base_lin_vel", valid_mask)
    ang_vel = as_array(df, "base_ang_vel", valid_mask)
    base_pose = as_array(df, "base_pose", valid_mask)
    proj_grav = as_array(df, "proj_grav", valid_mask)
    q_vel = as_array(df, "dof_vel", valid_mask)
    q_tau = as_array(df, "tau_act", valid_mask)
    grfs = as_array(df, "grf", valid_mask)
    ff_tau = as_array(df, "tau_ff", valid_mask)
    pd_tau = as_array(df, "tau_pd", valid_mask)

    if q_actions is not None and q_obs is not None and q_actions.shape == q_obs.shape:
        e = q_actions - q_obs
        metrics["dof_tracking_rmse"] = rmse(e)
        metrics["dof_tracking_mae"] = mae(e)
        metrics["q_action_norm_mean"] = mean_norm(q_actions)
        metrics["q_action_norm_std"] = std_norm(q_actions)
        if joint_limits is not None:
            jl = np.asarray(joint_limits)
            err = -(q_actions - jl[0, :]).clip(max=0.0)
            err += (q_actions - jl[1, :]).clip(min=0.0)
            metrics["joint_limit_violation_mean"] = float(np.mean(err))

    if vel_cmds is not None and lin_vel is not None and ang_vel is not None:
        lin_err = vel_cmds[:, 0:2] - lin_vel[:, 0:2]
        ang_err = vel_cmds[:, 2] - ang_vel[:, 2]
        cmd_err = np.concatenate((lin_err, ang_err[:, None]), axis=1)
        metrics["lin_cmd_rmse"] = rmse(lin_err)
        metrics["lin_cmd_mae"] = mae(lin_err)
        metrics["lin_cmd_std_sqroot"] = float(np.sqrt(np.std(np.square(lin_err))))
        metrics["ang_cmd_rmse"] = rmse(ang_err)
        metrics["ang_cmd_mae"] = mae(ang_err)
        metrics["ang_cmd_std_sqroot"] = float(np.sqrt(np.std(np.square(ang_err))))
        metrics["total_cmd_rmse"] = rmse(cmd_err)
        metrics["total_cmd_mae"] = mae(cmd_err)

    if base_pose is not None:
        h = base_height_target - base_pose[:, 2]
        metrics["height_rmse"] = rmse(h)
        metrics["height_mae"] = mae(h)

    if proj_grav is not None:
        orn = np.linalg.norm(proj_grav[:, 0:2], axis=1, ord=1)
        metrics["projected_gravity_rp_norm_mean"] = safe_mean(orn)
        metrics["projected_gravity_rp_norm_std"] = safe_std(orn)

    if lin_vel is not None and ang_vel is not None:
        z_vel = lin_vel[:, 2]
        av_rp = np.linalg.norm(ang_vel[:, 0:2], axis=1, ord=1)
        unwanted = np.concatenate((z_vel[:, None], ang_vel[:, 0:2]), axis=1)
        unwanted_norm = np.linalg.norm(unwanted, axis=1, ord=1)
        metrics["z_vel_rmse"] = rmse(z_vel)
        metrics["z_vel_mae"] = mae(z_vel)
        metrics["ang_vel_rp_norm_mean"] = safe_mean(av_rp)
        metrics["ang_vel_rp_norm_std"] = safe_std(av_rp)
        metrics["total_unwanted_vel_norm_mean"] = safe_mean(unwanted_norm)
        metrics["total_unwanted_vel_norm_std"] = safe_std(unwanted_norm)

    if q_vel is not None and q_tau is not None and q_vel.shape == q_tau.shape:
        power = q_vel * q_tau
        metrics["joint_power_norm_mean"] = mean_norm(power)
        metrics["joint_power_norm_std"] = std_norm(power)

    if grfs is not None:
        flat = grfs.reshape(grfs.shape[0], -1)
        metrics["grf_norm_mean"] = mean_norm(flat)
        metrics["grf_norm_std"] = std_norm(flat)
    if ff_tau is not None:
        metrics["ff_tau_norm_mean"] = mean_norm(ff_tau)
        metrics["ff_tau_norm_std"] = std_norm(ff_tau)
    if pd_tau is not None:
        metrics["pd_tau_norm_mean"] = mean_norm(pd_tau)
        metrics["pd_tau_norm_std"] = std_norm(pd_tau)

    if use_pact_metrics and ff_tau is not None and pd_tau is not None:
        total_tau = ff_tau + pd_tau
        metrics["total_tau_cmd_norm_mean"] = mean_norm(total_tau)
        metrics["total_tau_cmd_norm_std"] = std_norm(total_tau)
        ff_norm = np.linalg.norm(ff_tau, axis=1)
        pd_norm = np.linalg.norm(pd_tau, axis=1)
        metrics["ff_tau_ratio_mean"] = safe_mean(ff_norm / (ff_norm + pd_norm + eps))
        metrics["pd_tau_ratio_mean"] = safe_mean(pd_norm / (ff_norm + pd_norm + eps))
        metrics["pd_to_ff_tau_norm_ratio"] = float(np.mean(pd_norm) / (np.mean(ff_norm) + eps))
        if joint_torque_limits is not None:
            lim = np.asarray(joint_torque_limits)
            violation = -(total_tau - (-lim)).clip(max=0.0)
            violation += (total_tau - lim).clip(min=0.0)
            metrics["joint_torque_limit_violation_mean"] = float(np.mean(violation))

    if use_pact_metrics and ff_tau is not None and pd_tau is not None and q_vel is not None:
        ff_power = ff_tau * q_vel
        pd_power = pd_tau * q_vel
        total_power = (ff_tau + pd_tau) * q_vel
        dot = np.sum(ff_power * pd_power, axis=1)
        ffn = np.linalg.norm(ff_power, axis=1)
        pdn = np.linalg.norm(pd_power, axis=1)
        cos = dot / ((ffn * pdn) + eps)
        metrics["ff_power_norm_mean"] = safe_mean(ffn)
        metrics["pd_power_norm_mean"] = safe_mean(pdn)
        metrics["pd_to_ff_power_ratio"] = float(np.mean(pdn) / (np.mean(ffn) + eps))
        metrics["power_alignment_mean"] = safe_mean(cos)
        metrics["power_alignment_std"] = safe_std(cos)
        metrics["fraction_antagonistic_energy"] = float(np.sum(np.maximum(-dot, 0.0)) / (np.sum(np.abs(dot)) + eps))
        numerator = np.abs(ff_power) + np.abs(pd_power) - np.abs(total_power)
        denominator = np.abs(ff_power) + np.abs(pd_power)
        metrics["internal_power_cancellation"] = float(np.mean(numerator) / (np.mean(denominator) + eps))
    return metrics

## New episode and disturbance-response metrics

In [ ]:
def _failure_series(df):
    if "failure_reset" in df.columns:
        return df["failure_reset"].astype(float)
    if "failure" in df.columns:
        return df["failure"].astype(float)
    return pd.Series(np.zeros(len(df)), index=df.index)


def _timeout_series(df):
    if "time_out" in df.columns:
        return df["time_out"].astype(float)
    return pd.Series(np.zeros(len(df)), index=df.index)


def _non_failure_series(df):
    if "non_failure_reset" in df.columns:
        return df["non_failure_reset"].astype(float)
    return pd.Series(np.zeros(len(df)), index=df.index)


def _episode_sort_cols(df):
    """Columns that define the temporal order within one episode."""
    cols = []
    if "episode_step" in df.columns:
        cols.append("episode_step")
    if "global_step" in df.columns:
        cols.append("global_step")
    return cols


def _sort_episode_frame(ep_df):
    """
    Sort one episode into temporal order.

    The raw CSV may interleave rows from many vectorized env episodes. Any
    metric using temporal differences/windows must therefore be computed only
    after grouping by episode and sorting within that episode.
    """
    sort_cols = _episode_sort_cols(ep_df)
    if sort_cols:
        return ep_df.sort_values(sort_cols, kind="mergesort").reset_index(drop=True)
    return ep_df.reset_index(drop=True)


def add_rowwise_new_metrics(tmp):
    tmp = tmp.copy()

    q_des = as_array(tmp, "q_des")
    q = as_array(tmp, "dof_pose")
    if q_des is not None and q is not None and q_des.shape == q.shape:
        tmp["dof_tracking_mae_row"] = np.mean(np.abs(q_des - q), axis=1)

    base_pose = as_array(tmp, "base_pose")
    if base_pose is not None and base_pose.ndim == 2 and base_pose.shape[1] >= 3:
        tmp["base_x"] = base_pose[:, 0]
        tmp["base_y"] = base_pose[:, 1]
        tmp["base_z"] = base_pose[:, 2]
        tmp["height_abs_error_row"] = np.abs(BASE_HEIGHT_TARGET - base_pose[:, 2])

    base_rpy = as_array(tmp, "base_rpy")
    if base_rpy is not None and base_rpy.ndim == 2 and base_rpy.shape[1] >= 3:
        yaw = wrap_to_pi(base_rpy[:, 2])
        tmp["heading_error"] = np.abs(wrap_to_pi(yaw - 0.0))
        tmp["heading_error_deg"] = np.degrees(tmp["heading_error"])

    vel_cmd = as_array(tmp, "base_cmd")
    lin_vel = as_array(tmp, "base_lin_vel")
    ang_vel = as_array(tmp, "base_ang_vel")
    if vel_cmd is not None and lin_vel is not None and ang_vel is not None:
        tmp["lin_cmd_l1_error_row"] = np.linalg.norm(vel_cmd[:, 0:2] - lin_vel[:, 0:2], axis=1, ord=1)
        tmp["forward_cmd_abs_error_row"] = np.abs(vel_cmd[:, 0] - lin_vel[:, 0])
        tmp["lateral_vel_abs_row"] = np.abs(lin_vel[:, 1])
        tmp["yaw_rate_abs_error_row"] = np.abs(vel_cmd[:, 2] - ang_vel[:, 2])

    # These are row-wise values, so they are safe to compute before episode grouping.
    # Temporal derivatives/events, such as push_change, are computed later inside
    # each sorted episode to avoid false changes caused by interleaved CSV rows.
    rand_push = as_array(tmp, "rand_push")
    if rand_push is not None and rand_push.ndim == 2:
        tmp["rand_push_l1"] = np.linalg.norm(rand_push, axis=1, ord=1)
        if rand_push.shape[1] >= 2:
            tmp["rand_push_y"] = rand_push[:, 1]
    tmp["push_change"] = 0

    return tmp


def add_episode_push_change(ep_df):
    """
    Detect push resampling events within a single temporally sorted episode.

    This must not be run on the full dataframe, because rows from vectorized
    environments are interleaved. Running np.diff(rand_push) globally would
    compare unrelated episodes/envs and create false push events.
    """
    ep_df = _sort_episode_frame(ep_df)
    ep_df["push_change"] = 0

    rand_push = as_array(ep_df, "rand_push")
    if rand_push is None or rand_push.ndim != 2 or len(ep_df) <= 1:
        return ep_df

    delta = np.zeros(len(ep_df), dtype=float)
    delta[1:] = np.linalg.norm(np.diff(rand_push, axis=0), axis=1, ord=1)
    mag = np.linalg.norm(rand_push, axis=1, ord=1)

    # Do not mark the first row as a push event. It may inherit a sampled push
    # value from initialization/reset, but the recovery window needs a prior
    # within-episode transition and a stable y0 reference.
    ep_df["push_change"] = ((delta > PUSH_CHANGE_TOL) & (mag > MIN_PUSH_MAG)).astype(int)
    return ep_df


def compute_lateral_push_events(ep_df, episode_id, approach, terrain, disturbance_condition, source_file):
    ep_df = add_episode_push_change(ep_df)
    if "base_y" not in ep_df.columns or "push_change" not in ep_df.columns:
        return pd.DataFrame()

    ignore_steps = max(0, int(round(IGNORE_TIME / CONTROL_DT)))
    recovery_steps = max(ignore_steps + 1, int(round(RECOVERY_TIME / CONTROL_DT)))
    push_indices = np.flatnonzero(ep_df["push_change"].to_numpy(dtype=int) > 0)
    rows = []

    y = ep_df["base_y"].to_numpy(dtype=float)
    ep_steps = ep_df["episode_step"].to_numpy(dtype=float) if "episode_step" in ep_df.columns else np.arange(len(ep_df))
    push_y_signal = ep_df["rand_push_y"].to_numpy(dtype=float) if "rand_push_y" in ep_df.columns else np.full(len(ep_df), np.nan)

    for local_idx in push_indices:
        start = local_idx + ignore_steps
        end = min(local_idx + recovery_steps + 1, len(ep_df))
        if start >= len(ep_df) or end <= start:
            continue
        y0 = y[local_idx]
        drift = np.abs(y[start:end] - y0)
        rows.append({
            "approach": approach,
            "terrain": terrain,
            "disturbance_condition": disturbance_condition,
            "source_file": source_file,
            "episode": episode_id,
            "push_episode_step": ep_steps[local_idx],
            "push_local_index": int(local_idx),
            "push_y": float(push_y_signal[local_idx]) if np.isfinite(push_y_signal[local_idx]) else np.nan,
            "ignore_time_s": IGNORE_TIME,
            "recovery_time_s": RECOVERY_TIME,
            "lateral_drift_peak": float(np.nanmax(drift)),
            "lateral_drift_auc": float(np.nanmean(drift)),
            "lateral_drift_final": float(drift[-1]),
            "window_num_rows": int(len(drift)),
        })
    return pd.DataFrame(rows)


def compute_episode_results(df, approach, terrain, disturbance_condition, source_file):
    if "episode" not in df.columns:
        df = df.copy()
        df["episode"] = 0
    tmp = add_rowwise_new_metrics(df)
    tmp["failure"] = _failure_series(tmp)
    tmp["time_out"] = _timeout_series(tmp)
    tmp["non_failure_reset"] = _non_failure_series(tmp)

    episode_rows, push_event_chunks, sorted_episode_chunks = [], [], []
    for episode_id, ep_df in tmp.groupby("episode", sort=True):
        ep_df = add_episode_push_change(ep_df)
        sorted_episode_chunks.append(ep_df)

        failure = float(ep_df["failure"].max())
        timeout = float(ep_df["time_out"].max())
        non_failure = float(ep_df["non_failure_reset"].max())
        done = float(ep_df["done"].max()) if "done" in ep_df.columns else max(failure, timeout, non_failure)
        success = 1.0 if (non_failure > 0 or (done > 0 and failure == 0 and timeout == 0)) else 0.0
        episode_length = float(ep_df["episode_step"].max()) if "episode_step" in ep_df.columns else float(len(ep_df))

        row = {
            "approach": approach,
            "terrain": terrain,
            "disturbance_condition": disturbance_condition,
            "source_file": source_file,
            "episode": episode_id,
            "success": success,
            "failure": failure,
            "time_out": timeout,
            "non_failure_reset": non_failure,
            "episode_length": episode_length,
            "completion_time_s": episode_length * CONTROL_DT if success > 0 else np.nan,
            "termination_time_s": episode_length * CONTROL_DT,
        }
        for col in [
            "dof_tracking_mae_row", "height_abs_error_row", "heading_error", "heading_error_deg",
            "lin_cmd_l1_error_row", "forward_cmd_abs_error_row", "lateral_vel_abs_row", "yaw_rate_abs_error_row",
        ]:
            if col in ep_df.columns:
                out_col = col.replace("_row", "")
                row[out_col + "_mean"] = float(ep_df[col].mean())
                row[out_col + "_max"] = float(ep_df[col].max())

        push_events = compute_lateral_push_events(ep_df, episode_id, approach, terrain, disturbance_condition, source_file)
        if len(push_events):
            push_event_chunks.append(push_events)
            row["num_push_events"] = int(len(push_events))
            row["lateral_drift_peak_mean"] = float(push_events["lateral_drift_peak"].mean())
            row["lateral_drift_peak_max"] = float(push_events["lateral_drift_peak"].max())
            row["lateral_drift_auc_mean"] = float(push_events["lateral_drift_auc"].mean())
            row["lateral_drift_final_mean"] = float(push_events["lateral_drift_final"].mean())
        else:
            row["num_push_events"] = 0
            row["lateral_drift_peak_mean"] = np.nan
            row["lateral_drift_peak_max"] = np.nan
            row["lateral_drift_auc_mean"] = np.nan
            row["lateral_drift_final_mean"] = np.nan
        episode_rows.append(row)

    per_episode = pd.DataFrame(episode_rows)
    push_events = pd.concat(push_event_chunks, ignore_index=True) if push_event_chunks else pd.DataFrame()
    metric_df = pd.concat(sorted_episode_chunks, ignore_index=True) if sorted_episode_chunks else tmp
    return per_episode, push_events, metric_df

## Compact rollout export utilities

In [ ]:
def select_rollout_episodes(per_episode, max_per_file=1):
    if max_per_file is None or max_per_file <= 0 or len(per_episode) == 0:
        return []
    choices = []
    succ = per_episode[per_episode["success"] > 0]
    if len(succ):
        choices.extend(succ["episode"].head(max_per_file).tolist())
    if len(choices) < max_per_file:
        rest = per_episode[~per_episode["episode"].isin(choices)]
        choices.extend(rest["episode"].head(max_per_file - len(choices)).tolist())
    return choices


def compact_rollout_rows(metric_df, per_episode, approach, terrain, disturbance_condition, source_file):
    episodes = select_rollout_episodes(per_episode, ROLLOUT_EPISODES_PER_FILE)
    if not episodes:
        return pd.DataFrame()
    keep = metric_df[metric_df["episode"].isin(episodes)].copy()
    sort_cols = [c for c in ["episode", "episode_step", "global_step"] if c in keep.columns]
    if sort_cols:
        keep = keep.sort_values(sort_cols, kind="mergesort")
    if ROLLOUT_MAX_ROWS_PER_EPISODE is not None:
        keep = keep.groupby("episode", group_keys=False).head(ROLLOUT_MAX_ROWS_PER_EPISODE)

    cols = [
        "episode", "episode_step", "global_step", "done", "failure", "time_out", "non_failure_reset",
        "base_x", "base_y", "base_z", "heading_error", "heading_error_deg",
        "dof_tracking_mae_row", "height_abs_error_row", "lin_cmd_l1_error_row",
        "forward_cmd_abs_error_row", "lateral_vel_abs_row", "yaw_rate_abs_error_row",
        "push_change", "rand_push_l1", "rand_push_y",
    ]
    cols = [c for c in cols if c in keep.columns]
    out = keep[cols].copy()
    out["time_s"] = out["episode_step"].astype(float) * CONTROL_DT if "episode_step" in out.columns else np.arange(len(out)) * CONTROL_DT
    out["approach"] = approach
    out["terrain"] = terrain
    out["disturbance_condition"] = disturbance_condition
    out["source_file"] = source_file
    out["approach_label"] = _display_approach_name(approach)
    return out

## File-wise processing

In [ ]:
def process_one_file(row):
    path = Path(row["path"])
    approach = row["approach"]
    terrain = row["terrain"]
    disturbance_condition = row["disturbance_condition"]

    df = load_eval_csv(path)
    df = df.loc[get_row_mask_for_metrics(df)].reset_index(drop=True)

    scalar_metrics = compute_eval_metrics(
        df,
        approach=approach,
        base_height_target=BASE_HEIGHT_TARGET,
        joint_limits=JOINT_LIMITS,
        joint_torque_limits=JOINT_TORQUE_LIMITS,
        eps=EPS,
    )
    per_episode, push_events, metric_df = compute_episode_results(df, approach, terrain, disturbance_condition, str(path))
    rollout = compact_rollout_rows(metric_df, per_episode, approach, terrain, disturbance_condition, str(path)) if EXPORT_ROLLOUTS else pd.DataFrame()

    file_metrics = dict(row)
    file_metrics.update(scalar_metrics)
    file_metrics.update({
        "num_episodes": int(len(per_episode)),
        "episode_success_rate": float(per_episode["success"].mean()) if len(per_episode) else np.nan,
        "episode_failure_rate": float(per_episode["failure"].mean()) if len(per_episode) else np.nan,
        "episode_timeout_rate": float(per_episode["time_out"].mean()) if len(per_episode) else np.nan,
        "episode_non_failure_reset_rate": float(per_episode["non_failure_reset"].mean()) if len(per_episode) else np.nan,
        "completion_time_mean_s": float(per_episode["completion_time_s"].mean()) if "completion_time_s" in per_episode else np.nan,
        "heading_error_mean": float(per_episode["heading_error_mean"].mean()) if "heading_error_mean" in per_episode else np.nan,
        "heading_error_deg_mean": float(per_episode["heading_error_deg_mean"].mean()) if "heading_error_deg_mean" in per_episode else np.nan,
        "lateral_drift_peak_mean": float(per_episode["lateral_drift_peak_mean"].mean()) if "lateral_drift_peak_mean" in per_episode else np.nan,
        "lateral_drift_auc_mean": float(per_episode["lateral_drift_auc_mean"].mean()) if "lateral_drift_auc_mean" in per_episode else np.nan,
    })
    del df, metric_df
    gc.collect()
    return file_metrics, per_episode, push_events, rollout


def run_combined_analysis(file_table, results_dir=RESULTS_DIR):
    file_metrics_path = results_dir / "per_file_metrics.csv"
    episode_results_path = results_dir / "per_episode_results.csv"
    push_events_path = results_dir / "push_event_results.csv"
    rollout_path = results_dir / "selected_rollout_metrics.csv"

    if ((not RECOMPUTE_METRICS) and file_metrics_path.exists() and episode_results_path.exists() and push_events_path.exists()):
        per_file = pd.read_csv(file_metrics_path)
        per_episode = pd.read_csv(episode_results_path)
        push_events = pd.read_csv(push_events_path)
        rollout = pd.read_csv(rollout_path) if rollout_path.exists() else pd.DataFrame()
        return per_file, per_episode, push_events, rollout

    metric_rows, episode_chunks, push_event_chunks, rollout_chunks = [], [], [], []
    table = file_table.reset_index(drop=True)
    for i, row in table.iterrows():
        print(f"[{i + 1}/{len(table)}] {row['path']}")
        m, ep, pe, ro = process_one_file(row)
        metric_rows.append(m)
        if ep is not None and len(ep): episode_chunks.append(ep)
        if pe is not None and len(pe): push_event_chunks.append(pe)
        if ro is not None and len(ro): rollout_chunks.append(ro)

        if (i + 1) % SAVE_EVERY == 0:
            pd.DataFrame(metric_rows).to_csv(file_metrics_path, index=False)
            if episode_chunks: pd.concat(episode_chunks, ignore_index=True).to_csv(episode_results_path, index=False)
            if push_event_chunks: pd.concat(push_event_chunks, ignore_index=True).to_csv(push_events_path, index=False)
            if rollout_chunks: pd.concat(rollout_chunks, ignore_index=True).to_csv(rollout_path, index=False)

    per_file = pd.DataFrame(metric_rows)
    per_episode = pd.concat(episode_chunks, ignore_index=True) if episode_chunks else pd.DataFrame()
    push_events = pd.concat(push_event_chunks, ignore_index=True) if push_event_chunks else pd.DataFrame()
    rollout = pd.concat(rollout_chunks, ignore_index=True) if rollout_chunks else pd.DataFrame()
    per_file.to_csv(file_metrics_path, index=False)
    per_episode.to_csv(episode_results_path, index=False)
    push_events.to_csv(push_events_path, index=False)
    rollout.to_csv(rollout_path, index=False)
    return per_file, per_episode, push_events, rollout

## Optional process-parallel runner
Use this if your notebook environment supports multiprocessing. If pickling errors occur, use the sequential runner above or export this notebook as a `.py` script.

In [ ]:
def _process_one_file_from_dict(row_dict):
    return process_one_file(pd.Series(row_dict))


def run_combined_analysis_parallel(file_table, results_dir=RESULTS_DIR, max_workers=MAX_WORKERS, save_every=SAVE_EVERY):
    file_metrics_path = results_dir / "per_file_metrics.csv"
    episode_results_path = results_dir / "per_episode_results.csv"
    push_events_path = results_dir / "push_event_results.csv"
    rollout_path = results_dir / "selected_rollout_metrics.csv"

    if ((not RECOMPUTE_METRICS) and file_metrics_path.exists() and episode_results_path.exists() and push_events_path.exists()):
        per_file = pd.read_csv(file_metrics_path)
        per_episode = pd.read_csv(episode_results_path)
        push_events = pd.read_csv(push_events_path)
        rollout = pd.read_csv(rollout_path) if rollout_path.exists() else pd.DataFrame()
        return per_file, per_episode, push_events, rollout

    rows = file_table.reset_index(drop=True).to_dict(orient="records")
    total = len(rows)
    if total == 0:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    if max_workers is None:
        max_workers = max(1, min(os.cpu_count() or 1, total))

    metric_rows, episode_chunks, push_event_chunks, rollout_chunks = [], [], [], []
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(_process_one_file_from_dict, row): row for row in rows}
        for completed, future in enumerate(as_completed(futures), start=1):
            row = futures[future]
            path = row.get("path", "<unknown path>")
            try:
                m, ep, pe, ro = future.result()
            except Exception as e:
                print(f"[{completed}/{total}] FAILED: {path}")
                print(f"    {type(e).__name__}: {e}")
                continue
            print(f"[{completed}/{total}] finished: {path}")
            metric_rows.append(m)
            if ep is not None and len(ep): episode_chunks.append(ep)
            if pe is not None and len(pe): push_event_chunks.append(pe)
            if ro is not None and len(ro): rollout_chunks.append(ro)
            if completed % save_every == 0:
                pd.DataFrame(metric_rows).to_csv(file_metrics_path, index=False)
                if episode_chunks: pd.concat(episode_chunks, ignore_index=True).to_csv(episode_results_path, index=False)
                if push_event_chunks: pd.concat(push_event_chunks, ignore_index=True).to_csv(push_events_path, index=False)
                if rollout_chunks: pd.concat(rollout_chunks, ignore_index=True).to_csv(rollout_path, index=False)

    per_file = pd.DataFrame(metric_rows)
    per_episode = pd.concat(episode_chunks, ignore_index=True) if episode_chunks else pd.DataFrame()
    push_events = pd.concat(push_event_chunks, ignore_index=True) if push_event_chunks else pd.DataFrame()
    rollout = pd.concat(rollout_chunks, ignore_index=True) if rollout_chunks else pd.DataFrame()
    per_file.to_csv(file_metrics_path, index=False)
    per_episode.to_csv(episode_results_path, index=False)
    push_events.to_csv(push_events_path, index=False)
    rollout.to_csv(rollout_path, index=False)
    return per_file, per_episode, push_events, rollout

## Run analysis

In [ ]:
# per_file_metrics, per_episode_results, push_event_results, selected_rollout_metrics = run_combined_analysis(
    # file_table,
    # results_dir=RESULTS_DIR,
# )

# Multiprocessing alternative:
per_file_metrics, per_episode_results, push_event_results, selected_rollout_metrics = run_combined_analysis_parallel(
    file_table, results_dir=RESULTS_DIR, max_workers=MAX_WORKERS, save_every=SAVE_EVERY
)

print(f"Per-file rows: {len(per_file_metrics)}")
print(f"Per-episode rows: {len(per_episode_results)}")
print(f"Push-event rows: {len(push_event_results)}")
print(f"Selected rollout rows: {len(selected_rollout_metrics)}")
per_file_metrics.head()

## Summary tables

In [ ]:
def _ordered_labels(labels):
    """
    Return labels sorted using APPROACH_ORDER first, then any remaining labels alphabetically.

    APPROACH_ORDER can contain either display labels like "PACT" or raw labels that
    get mapped through _display_approach_name().
    """
    labels = list(labels)

    # If APPROACH_ORDER is not defined, fall back to alphabetical order.
    if "APPROACH_ORDER" not in globals() or APPROACH_ORDER is None:
        return sorted(labels)

    preferred = []
    for item in APPROACH_ORDER:
        # Allow either raw approach names or already-display-formatted names.
        display_item = _display_approach_name(item)
        if display_item in labels and display_item not in preferred:
            preferred.append(display_item)
        elif item in labels and item not in preferred:
            preferred.append(item)

    remaining = sorted([x for x in labels if x not in preferred])
    return preferred + remaining

def _sort_with_preferred_order(df, label_col="approach_label", terrain_col="terrain"):
    """Sort a result table using APPROACH_ORDER while keeping terrain/disturbance grouping readable."""
    out = df.copy()
    order = _ordered_labels(out[label_col].dropna().unique()) if label_col in out.columns else []
    order_map = {label: i for i, label in enumerate(order)}
    if label_col in out.columns:
        out["_approach_order"] = out[label_col].map(order_map).fillna(len(order_map)).astype(int)
    else:
        out["_approach_order"] = 0

    sort_cols = []
    if terrain_col in out.columns:
        # Put concrete terrains first and ALL at the end.
        out["_terrain_is_all"] = (out[terrain_col] == "ALL").astype(int)
        sort_cols.extend(["_terrain_is_all", terrain_col])
    if "disturbance_condition" in out.columns:
        sort_cols.append("disturbance_condition")
    sort_cols.append("_approach_order")
    if label_col in out.columns:
        sort_cols.append(label_col)

    out = out.sort_values(sort_cols, ignore_index=True)
    return out.drop(columns=[c for c in ["_approach_order", "_terrain_is_all"] if c in out.columns])


def summarize_combined_results(per_episode_results, per_file_metrics, include_all_terrains=True):
    ep = per_episode_results.copy()
    if len(ep) == 0:
        return pd.DataFrame()
    ep["approach_label"] = ep["approach"].map(_display_approach_name)

    agg = dict(
        num_episodes=("episode", "count"),
        success_rate=("success", "mean"),
        failure_rate=("failure", "mean"),
        timeout_rate=("time_out", "mean"),
        non_failure_reset_rate=("non_failure_reset", "mean"),
        termination_time_s_mean=("termination_time_s", "mean"),
        completion_time_s_mean=("completion_time_s", "mean"),
        completion_time_s_std=("completion_time_s", "std"),
        heading_error_mean=("heading_error_mean", "mean"),
        heading_error_deg_mean=("heading_error_deg_mean", "mean"),
        lateral_drift_peak_mean=("lateral_drift_peak_mean", "mean"),
        lateral_drift_peak_max=("lateral_drift_peak_max", "max"),
        lateral_drift_auc_mean=("lateral_drift_auc_mean", "mean"),
        num_push_events=("num_push_events", "sum"),
    )
    for c in ["dof_tracking_mae_mean", "height_abs_error_mean", "lin_cmd_l1_error_mean", "forward_cmd_abs_error_mean"]:
        if c in ep.columns:
            agg[c] = (c, "mean")

    group_cols = ["approach", "approach_label", "terrain", "disturbance_condition"]
    per_terrain_summary = ep.groupby(group_cols, as_index=False).agg(**agg)

    # Add ALL-terrain rows by averaging over all episode-level outcomes for each approach/disturbance.
    summary_parts = [per_terrain_summary]
    if include_all_terrains:
        all_ep = ep.groupby(["approach", "approach_label", "disturbance_condition"], as_index=False).agg(**agg)
        all_ep["terrain"] = "ALL"
        all_ep = all_ep[["approach", "approach_label", "terrain", "disturbance_condition"] + [c for c in all_ep.columns if c not in ["approach", "approach_label", "terrain", "disturbance_condition"]]]
        summary_parts.append(all_ep)

    summary = pd.concat(summary_parts, ignore_index=True)

    metric_cols = [
        "lin_cmd_mae", "ang_cmd_mae", "total_cmd_mae", "height_mae", "dof_tracking_mae",
        "projected_gravity_rp_norm_mean", "z_vel_mae", "ang_vel_rp_norm_mean",
        "joint_power_norm_mean", "grf_norm_mean", "ff_tau_norm_mean", "pd_tau_norm_mean",
        "ff_tau_ratio_mean", "pd_to_ff_power_ratio", "power_alignment_mean",
        "fraction_antagonistic_energy", "internal_power_cancellation",
    ]
    metric_cols = [c for c in metric_cols if c in per_file_metrics.columns]
    if metric_cols and len(per_file_metrics):
        pf = per_file_metrics.copy()
        pf["approach_label"] = pf["approach"].map(_display_approach_name)

        file_summary = pf.groupby(group_cols, as_index=False)[metric_cols].mean()
        file_parts = [file_summary]
        if include_all_terrains:
            all_file_summary = pf.groupby(["approach", "approach_label", "disturbance_condition"], as_index=False)[metric_cols].mean()
            all_file_summary["terrain"] = "ALL"
            all_file_summary = all_file_summary[["approach", "approach_label", "terrain", "disturbance_condition"] + metric_cols]
            file_parts.append(all_file_summary)

        file_summary = pd.concat(file_parts, ignore_index=True)
        summary = summary.merge(file_summary, on=group_cols, how="left")

    return _sort_with_preferred_order(summary).reset_index(drop=True)


combined_summary = summarize_combined_results(per_episode_results, per_file_metrics, include_all_terrains=True)
combined_summary.to_csv(RESULTS_DIR / "combined_summary.csv", index=False)
combined_summary[combined_summary["terrain"].eq("ALL")].to_csv(RESULTS_DIR / "all_terrain_summary.csv", index=False)
combined_summary.head(20)

In [ ]:
compact_cols = [
    "approach_label", "terrain", "disturbance_condition", "num_episodes",
    "success_rate", "failure_rate", "completion_time_s_mean", "heading_error_deg_mean",
    "lateral_drift_peak_mean", "lateral_drift_auc_mean", "dof_tracking_mae", "height_mae", "total_cmd_mae",
]
compact_cols = [c for c in compact_cols if c in combined_summary.columns]
combined_summary[compact_cols].round(4)

## Plotting helpers

In [ ]:
def _normalize_approach_key(x):
    """Map raw approach ids and display labels to the display label used in plots."""
    if x is None:
        return None
    if x in APPROACH_DISPLAY_NAMES:
        return APPROACH_DISPLAY_NAMES[x]
    return x


def _ordered_labels(labels):
    labels = list(labels)
    if APPROACH_ORDER is None:
        return labels
    preferred = [_normalize_approach_key(a) for a in APPROACH_ORDER]
    ordered = [a for a in preferred if a in labels]
    ordered += [a for a in labels if a not in ordered]
    return ordered


def _highlight_label():
    return _normalize_approach_key(HIGHLIGHT_APPROACH)


def _bar_style(label):
    """Return styling kwargs for bars, emphasizing HIGHLIGHT_APPROACH."""
    is_highlight = label == _highlight_label()
    return {
        "alpha": 1.0 if is_highlight else 0.55,
        "edgecolor": "black" if is_highlight else None,
        "linewidth": 2.0 if is_highlight else 0.0,
        "zorder": 10 if is_highlight else 2,
    }


def _line_style(label, base_linewidth=1.4):
    """Return styling kwargs for lines, emphasizing HIGHLIGHT_APPROACH."""
    is_highlight = label == _highlight_label()
    return {
        "linewidth": 2.8 if is_highlight else base_linewidth,
        "markersize": 5.5 if is_highlight else 4.0,
        "alpha": 1.0 if is_highlight else 0.55,
        "zorder": 10 if is_highlight else 2,
    }


def _filter_summary(summary, terrain=None, disturbance_condition=None):
    df = summary.copy()
    if terrain is not None:
        df = df[df["terrain"] == terrain]
    if disturbance_condition is not None:
        df = df[df["disturbance_condition"] == disturbance_condition]
    return df

def save_paper_fig(fig, path, pad_inches=0.01):
    fig.savefig(
        path,
        bbox_inches="tight",
        pad_inches=pad_inches,
        dpi=300,
    )

def compactify_plot(
    fig,
    ax=None,
    pad=0.02,
    tick_pad=1,
    label_pad=1,
    title_pad=2,
):
    """
    Reduce whitespace around a matplotlib figure for paper-ready plots.
    """
    axes = fig.axes if ax is None else [ax]

    for a in axes:
        a.tick_params(axis="both", which="major", pad=tick_pad)
        a.xaxis.labelpad = label_pad
        a.yaxis.labelpad = label_pad

    fig.tight_layout(pad=pad)
    return fig

def plot_rate_bars(summary, terrain="ALL", disturbance_condition=None, approach_order=None, highlight=None):
    global APPROACH_ORDER, HIGHLIGHT_APPROACH
    old_order, old_highlight = APPROACH_ORDER, HIGHLIGHT_APPROACH
    if approach_order is not None:
        APPROACH_ORDER = approach_order
    if highlight is not None:
        HIGHLIGHT_APPROACH = highlight
    try:
        df = _filter_summary(summary, terrain=terrain, disturbance_condition=disturbance_condition)
        plot_df = df.groupby("approach_label", as_index=False)[["success_rate", "failure_rate"]].mean()
        order = _ordered_labels(plot_df["approach_label"])
        plot_df = plot_df.set_index("approach_label").reindex(order).dropna(how="all").reset_index()

        x = np.arange(len(plot_df))
        width = 0.38
        fig, ax = plt.subplots(figsize=FIGURE_SIZE)
        for xi, (_, row) in zip(x, plot_df.iterrows()):
            style = _bar_style(row["approach_label"])
            ax.bar(xi - width/2, 100 * row["success_rate"], width, label="Success" if xi == 0 else None, **style)
            ax.bar(xi + width/2, 100 * row["failure_rate"], width, label="Failure" if xi == 0 else None, **style)
        ax.set_ylabel("Episode Rate (%)")
        ax.set_xlabel("Approach")
        ax.set_xticks(x)
        ax.set_xticklabels(plot_df["approach_label"], rotation=30, ha="right")
        ax.set_ylim(0, 105)
        ax.grid(True, axis="y", alpha=0.3)
        ax.legend(frameon=False)
        fig.tight_layout()
        return fig, ax
    finally:
        APPROACH_ORDER, HIGHLIGHT_APPROACH = old_order, old_highlight


def plot_success_failure_rates(
    summary,
    terrain="ALL",
    approach_order=None,
    highlight="PC",
    ax=None,
    annotate=False,
):
    if ax is None:
        fig, ax = plt.subplots(figsize=FIGURE_SIZE)
    else:
        fig = ax.figure

    df = summary.copy()
    if terrain is not None and "terrain" in df.columns:
        df = df[df["terrain"].eq(terrain)].copy()

    if "approach_label" not in df.columns:
        df["approach_label"] = df["approach"].map(_display_approach_name)

    if approach_order is None:
        approach_order = _ordered_labels(df["approach_label"].dropna().unique())

    df = df.set_index("approach_label").reindex(approach_order).reset_index()
    df = df.dropna(subset=["success_rate", "failure_rate"], how="all")

    labels = df["approach_label"].tolist()
    x = np.arange(len(labels))
    width = 0.36

    success_vals = 100.0 * df["success_rate"].to_numpy()
    failure_vals = 100.0 * df["failure_rate"].to_numpy()

    success_bars = ax.bar(
        x - width / 2,
        success_vals,
        width,
        label="Success",
        color="tab:blue",
        alpha=0.85,
    )

    failure_bars = ax.bar(
        x + width / 2,
        failure_vals,
        width,
        label="Failure",
        color="tab:orange",
        alpha=0.85,
    )

    if annotate:
        ax.bar_label(
            success_bars,
            labels=[f"{v:.1f}" for v in success_vals],
            padding=2,
            fontsize=8,
        )
        ax.bar_label(
            failure_bars,
            labels=[f"{v:.1f}" for v in failure_vals],
            padding=2,
            fontsize=8,
        )

    # Highlight one approach by thickening both bars.
    if highlight is not None:
        highlight_label = _display_approach_name(highlight)
        for label, sbar, fbar in zip(labels, success_bars, failure_bars):
            if label == highlight_label or label == highlight:
                for bar in (sbar, fbar):
                    bar.set_edgecolor("black")
                    bar.set_linewidth(2.0)

    ax.set_xlabel("Approach")
    ax.set_ylabel("Episode Rate (%)")
    ax.set_ylim(0, 105)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=30, ha="right")
    ax.grid(True, axis="y", alpha=0.3)
    ax.legend(frameon=False)

    fig.tight_layout()
    return fig, ax

def plot_success_failure_rates_timeouts(
    summary,
    terrain="ALL",
    approach_order=None,
    highlight="PC",
    ax=None,
    annotate=False,
):
    if ax is None:
        fig, ax = plt.subplots(figsize=FIGURE_SIZE)
    else:
        fig = ax.figure

    df = summary.copy()

    if terrain is not None and "terrain" in df.columns:
        df = df[df["terrain"].eq(terrain)].copy()

    if "approach_label" not in df.columns:
        df["approach_label"] = df["approach"].map(_display_approach_name)

    if "timeout_rate" not in df.columns:
        df["timeout_rate"] = 0.0

    if approach_order is None:
        approach_order = _ordered_labels(df["approach_label"].dropna().unique())

    df = df.set_index("approach_label").reindex(approach_order).reset_index()
    df = df.dropna(subset=["success_rate", "failure_rate", "timeout_rate"], how="all")

    labels = df["approach_label"].tolist()
    x = np.arange(len(labels))
    width = 0.26

    success_vals = 100.0 * df["success_rate"].fillna(0.0).to_numpy()
    failure_vals = 100.0 * df["failure_rate"].fillna(0.0).to_numpy()
    timeout_vals = 100.0 * df["timeout_rate"].fillna(0.0).to_numpy()

    success_bars = ax.bar(
        x - width,
        success_vals,
        width,
        label="Success",
        color="tab:blue",
        alpha=0.85,
    )

    failure_bars = ax.bar(
        x,
        failure_vals,
        width,
        label="Failure",
        color="tab:orange",
        alpha=0.85,
    )

    timeout_bars = ax.bar(
        x + width,
        timeout_vals,
        width,
        label="Timeout",
        color="tab:gray",
        alpha=0.85,
    )

    if annotate:
        ax.bar_label(
            success_bars,
            labels=[f"{v:.1f}" if v > 0 else "" for v in success_vals],
            padding=2,
            fontsize=8,
        )
        ax.bar_label(
            failure_bars,
            labels=[f"{v:.1f}" if v > 0 else "" for v in failure_vals],
            padding=2,
            fontsize=8,
        )
        ax.bar_label(
            timeout_bars,
            labels=[f"{v:.1f}" if v > 0 else "" for v in timeout_vals],
            padding=2,
            fontsize=8,
        )

    # Highlight one approach by thickening all three bars.
    if highlight is not None:
        highlight_label = _display_approach_name(highlight)

        for label, sbar, fbar, tbar in zip(
            labels,
            success_bars,
            failure_bars,
            timeout_bars,
        ):
            if label == highlight_label or label == highlight:
                for bar in (sbar, fbar, tbar):
                    bar.set_edgecolor("black")
                    bar.set_linewidth(2.0)

    ax.set_xlabel("Approach")
    ax.set_ylabel("Episode Rate (%)")
    ax.set_ylim(0, 110 if annotate else 105)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=30, ha="right")
    ax.grid(True, axis="y", alpha=0.3)
    ax.legend(frameon=False)

    fig.tight_layout()
    return fig, ax


def plot_completion_time(
    per_episode,
    terrain=None,
    disturbance_condition=None,
    plot_kind="box",
    approach_order=None,
    highlight=None,
    annotate=False,
):
    global APPROACH_ORDER, HIGHLIGHT_APPROACH

    old_order, old_highlight = APPROACH_ORDER, HIGHLIGHT_APPROACH

    if approach_order is not None:
        APPROACH_ORDER = approach_order
    if highlight is not None:
        HIGHLIGHT_APPROACH = highlight

    try:
        df = per_episode.copy()
        df["approach_label"] = df["approach"].map(_display_approach_name)
        df = df[df["success"] > 0]

        if terrain is not None:
            df = df[df["terrain"] == terrain]
        if disturbance_condition is not None:
            df = df[df["disturbance_condition"] == disturbance_condition]

        order = _ordered_labels(df["approach_label"].dropna().unique())

        fig, ax = plt.subplots(figsize=FIGURE_SIZE)

        if len(df) == 0:
            ax.text(0.5, 0.5, "No successful episodes", ha="center", va="center")
            return fig, ax

        data = [
            df.loc[df["approach_label"] == a, "completion_time_s"]
            .dropna()
            .to_numpy()
            for a in order
        ]

        means = [np.nanmean(d) if len(d) else np.nan for d in data]
        medians = [np.nanmedian(d) if len(d) else np.nan for d in data]

        if plot_kind == "bar":
            bars = ax.bar(order, means)

            for bar, label in zip(bars, order):
                style = _bar_style(label)
                bar.set_alpha(style["alpha"])

                if style["edgecolor"] is not None:
                    bar.set_edgecolor(style["edgecolor"])

                bar.set_linewidth(style["linewidth"])
                bar.set_zorder(style["zorder"])

            if annotate:
                ax.bar_label(
                    bars,
                    labels=[
                        f"{v:.2f}" if np.isfinite(v) else ""
                        for v in means
                    ],
                    padding=3,
                    fontsize=8,
                )

        else:
            try:
                bp = ax.boxplot(
                    data,
                    tick_labels=order,
                    showfliers=False,
                    patch_artist=True,
                )
            except TypeError:
                bp = ax.boxplot(
                    data,
                    labels=order,
                    showfliers=False,
                    patch_artist=True,
                )

            for patch, label in zip(bp["boxes"], order):
                style = _bar_style(label)
                patch.set_alpha(style["alpha"])

                if style["edgecolor"] is not None:
                    patch.set_edgecolor(style["edgecolor"])

                patch.set_linewidth(max(style["linewidth"], 1.0))

            for med in bp["medians"]:
                med.set_linewidth(1.6)

            if annotate:
                annotation_ys = []

                for i, (mean_val, median_val) in enumerate(zip(means, medians), start=1):
                    if not np.isfinite(mean_val) and not np.isfinite(median_val):
                        continue

                    upper_whisker = bp["whiskers"][2 * (i - 1) + 1]
                    y_anchor = np.max(upper_whisker.get_ydata())

                    ax.annotate(
                        f"μ={mean_val:.2f}\nmed={median_val:.2f}",
                        xy=(i, y_anchor),
                        xytext=(0, 8),
                        textcoords="offset points",
                        ha="center",
                        va="bottom",
                        fontsize=8,
                        clip_on=False,
                        bbox=dict(
                            boxstyle="round,pad=0.2",
                            facecolor="white",
                            edgecolor="none",
                            alpha=0.75,
                        ),
                    )

                    annotation_ys.append(y_anchor)

                if annotation_ys:
                    ymin, ymax = ax.get_ylim()
                    y_range = ymax - ymin
                    ax.set_ylim(ymin, max(ymax, max(annotation_ys) + 0.20 * y_range))

        ax.set_ylabel("Completion Time (s)")
        ax.set_xlabel("Approach")
        ax.tick_params(axis="x", rotation=30)
        ax.grid(True, axis="y", alpha=0.3)

        # Give annotations room at the top.
        # ymin, ymax = ax.get_ylim()
        # ax.set_ylim(ymin, ymax * 1.12)

        fig.tight_layout()
        return fig, ax

    finally:
        APPROACH_ORDER, HIGHLIGHT_APPROACH = old_order, old_highlight


def plot_metric_bar(summary, metric_col, ylabel, terrain="ALL", disturbance_condition=None, approach_order=None, highlight=None):
    global APPROACH_ORDER, HIGHLIGHT_APPROACH
    old_order, old_highlight = APPROACH_ORDER, HIGHLIGHT_APPROACH
    if approach_order is not None:
        APPROACH_ORDER = approach_order
    if highlight is not None:
        HIGHLIGHT_APPROACH = highlight
    try:
        df = _filter_summary(summary, terrain=terrain, disturbance_condition=disturbance_condition)
        plot_df = df.groupby("approach_label", as_index=False)[metric_col].mean()
        order = _ordered_labels(plot_df["approach_label"])
        plot_df = plot_df.set_index("approach_label").reindex(order).dropna(how="all").reset_index()
        fig, ax = plt.subplots(figsize=FIGURE_SIZE)
        bars = ax.bar(plot_df["approach_label"], plot_df[metric_col])
        for bar, label in zip(bars, plot_df["approach_label"]):
            style = _bar_style(label)
            bar.set_alpha(style["alpha"])
            if style["edgecolor"] is not None:
                bar.set_edgecolor(style["edgecolor"])
            bar.set_linewidth(style["linewidth"])
            bar.set_zorder(style["zorder"])
        ax.set_ylabel(ylabel)
        ax.set_xlabel("Approach")
        ax.tick_params(axis="x", rotation=30)
        ax.grid(True, axis="y", alpha=0.3)
        fig.tight_layout()
        return fig, ax
    finally:
        APPROACH_ORDER, HIGHLIGHT_APPROACH = old_order, old_highlight


def plot_metric_bar_box(
    summary,
    metric_col,
    ylabel,
    terrain="ALL",
    disturbance_condition=None,
    approach_order=None,
    highlight=None,
    plot_kind="bar",      # "bar" or "box"
    annotate=True,
):
    global APPROACH_ORDER, HIGHLIGHT_APPROACH

    old_order, old_highlight = APPROACH_ORDER, HIGHLIGHT_APPROACH

    if approach_order is not None:
        APPROACH_ORDER = approach_order
    if highlight is not None:
        HIGHLIGHT_APPROACH = highlight

    try:
        df = _filter_summary(
            summary,
            terrain=terrain,
            disturbance_condition=disturbance_condition,
        ).copy()

        if len(df) == 0 or metric_col not in df.columns:
            fig, ax = plt.subplots(figsize=FIGURE_SIZE)
            ax.text(
                0.5,
                0.5,
                f"No data for {metric_col}",
                ha="center",
                va="center",
            )
            return fig, ax

        if "approach_label" not in df.columns:
            df["approach_label"] = df["approach"].map(_display_approach_name)

        order = _ordered_labels(df["approach_label"].dropna().unique())

        fig, ax = plt.subplots(figsize=FIGURE_SIZE)

        if plot_kind == "bar":
            plot_df = (
                df.groupby("approach_label", as_index=False)[metric_col]
                .mean()
            )

            plot_df = (
                plot_df.set_index("approach_label")
                .reindex(order)
                .dropna(how="all")
                .reset_index()
            )

            labels = plot_df["approach_label"].tolist()
            vals = plot_df[metric_col].to_numpy(dtype=float)

            bars = ax.bar(labels, vals)

            for bar, label in zip(bars, labels):
                style = _bar_style(label)
                bar.set_alpha(style["alpha"])

                if style["edgecolor"] is not None:
                    bar.set_edgecolor(style["edgecolor"])

                bar.set_linewidth(style["linewidth"])
                bar.set_zorder(style["zorder"])

            if annotate:
                ax.bar_label(
                    bars,
                    labels=[
                        f"{v:.3f}" if np.isfinite(v) else ""
                        for v in vals
                    ],
                    padding=3,
                    fontsize=8,
                )

                ymin, ymax = ax.get_ylim()
                y_range = ymax - ymin
                ax.set_ylim(ymin, ymax + 0.12 * y_range)

        elif plot_kind == "box":
            data = [
                df.loc[df["approach_label"] == label, metric_col]
                .dropna()
                .to_numpy(dtype=float)
                for label in order
            ]

            # Remove approaches with no data.
            keep = [i for i, d in enumerate(data) if len(d) > 0]
            data = [data[i] for i in keep]
            labels = [order[i] for i in keep]

            if len(data) == 0:
                ax.text(
                    0.5,
                    0.5,
                    f"No valid data for {metric_col}",
                    ha="center",
                    va="center",
                )
                return fig, ax

            try:
                bp = ax.boxplot(
                    data,
                    tick_labels=labels,
                    showfliers=False,
                    patch_artist=True,
                )
            except TypeError:
                bp = ax.boxplot(
                    data,
                    labels=labels,
                    showfliers=False,
                    patch_artist=True,
                )

            for patch, label in zip(bp["boxes"], labels):
                style = _bar_style(label)
                patch.set_alpha(style["alpha"])

                if style["edgecolor"] is not None:
                    patch.set_edgecolor(style["edgecolor"])

                patch.set_linewidth(max(style["linewidth"], 1.0))

            for med in bp["medians"]:
                med.set_linewidth(1.6)

            means = [np.nanmean(d) if len(d) else np.nan for d in data]
            medians = [np.nanmedian(d) if len(d) else np.nan for d in data]

            if annotate:
                annotation_ys = []

                for i, (mean_val, median_val) in enumerate(zip(means, medians), start=1):
                    if not np.isfinite(mean_val) and not np.isfinite(median_val):
                        continue

                    # Use upper whisker, not max(data), because fliers are hidden.
                    upper_whisker = bp["whiskers"][2 * (i - 1) + 1]
                    y_anchor = np.max(upper_whisker.get_ydata())

                    ax.annotate(
                        f"μ={mean_val:.3f}\nmed={median_val:.3f}",
                        xy=(i, y_anchor),
                        xytext=(0, 8),
                        textcoords="offset points",
                        ha="center",
                        va="bottom",
                        fontsize=8,
                        clip_on=False,
                        bbox=dict(
                            boxstyle="round,pad=0.2",
                            facecolor="white",
                            edgecolor="none",
                            alpha=0.75,
                        ),
                    )

                    annotation_ys.append(y_anchor)

                if annotation_ys:
                    ymin, ymax = ax.get_ylim()
                    y_range = ymax - ymin
                    ax.set_ylim(
                        ymin,
                        max(ymax, max(annotation_ys) + 0.20 * y_range),
                    )

        else:
            raise ValueError(f"Unsupported plot_kind={plot_kind}. Use 'bar' or 'box'.")

        ax.set_ylabel(ylabel)
        ax.set_xlabel("Approach")
        ax.tick_params(axis="x", rotation=30)
        ax.grid(True, axis="y", alpha=0.3)

        fig.tight_layout()
        return fig, ax

    finally:
        APPROACH_ORDER, HIGHLIGHT_APPROACH = old_order, old_highlight

In [ ]:
per_episode_results.columns

## Main comparison plots

In [ ]:
PLOT_TERRAIN = "ALL"
PLOT_DISTURBANCE = None

# fig, ax = plot_rate_bars(combined_summary, terrain=PLOT_TERRAIN, disturbance_condition=PLOT_DISTURBANCE)
fig, ax = plot_success_failure_rates(combined_summary, terrain=PLOT_TERRAIN, annotate=True)
fig.savefig(RESULTS_DIR / "success_failure_rate_bars.png", dpi=300, bbox_inches="tight")
plt.show()

if SAVE_AS_PDF:
    compactify_plot(fig, ax)
    save_paper_fig(fig, RESULTS_DIR / "success_failure_rate_bars.pdf")


fig, ax = plot_success_failure_rates_timeouts(combined_summary, terrain=PLOT_TERRAIN, annotate=True)
fig.savefig(RESULTS_DIR / "success_failure_timeout_rate_bars.png", dpi=300, bbox_inches="tight")
plt.show()

if SAVE_AS_PDF:
    compactify_plot(fig, ax)
    save_paper_fig(fig, RESULTS_DIR / "success_failure_timeout_rate_bars.pdf")

fig, ax = plot_completion_time(per_episode_results, disturbance_condition=PLOT_DISTURBANCE, plot_kind="box", annotate=True)
fig.savefig(RESULTS_DIR / "completion_time_boxplot.png", dpi=300, bbox_inches="tight")
plt.show()

if SAVE_AS_PDF:
    compactify_plot(fig, ax)
    save_paper_fig(fig, RESULTS_DIR / "completion_time_boxplot.pdf")

# fig, ax = plot_completion_time(per_episode_results, disturbance_condition=PLOT_DISTURBANCE, plot_kind="bar", annotate=True)
# fig.savefig(RESULTS_DIR / "completion_time_barplot.png", dpi=300, bbox_inches="tight")
# plt.show()

In [ ]:
if "heading_error_deg_mean" in combined_summary.columns:
    fig, ax = plot_metric_bar_box(combined_summary, "heading_error_deg_mean", "Mean Heading Error (deg)", terrain=PLOT_TERRAIN, disturbance_condition=PLOT_DISTURBANCE, plot_kind="bar", annotate=True)
    fig.savefig(RESULTS_DIR / "heading_error_barplot.png", dpi=300, bbox_inches="tight")
    plt.show()

    fig, ax = plot_metric_bar_box(combined_summary, "heading_error_deg_mean", "Mean Heading Error (deg)", terrain=None, disturbance_condition=PLOT_DISTURBANCE, plot_kind="box", annotate=True)
    fig.savefig(RESULTS_DIR / "heading_error_boxplot.png", dpi=300, bbox_inches="tight")
    plt.show()

    if SAVE_AS_PDF:
        compactify_plot(fig, ax)
        save_paper_fig(fig, RESULTS_DIR / "heading_error_boxplot.pdf")

if "lateral_drift_peak_mean" in combined_summary.columns:
    fig, ax = plot_metric_bar_box(combined_summary, "lateral_drift_peak_mean", "Mean Peak Lateral Drift (m)", terrain=PLOT_TERRAIN, disturbance_condition=PLOT_DISTURBANCE, plot_kind="bar", annotate=True)
    fig.savefig(RESULTS_DIR / "lateral_drift_peak_barplot.png", dpi=300, bbox_inches="tight")
    plt.show()

    fig, ax = plot_metric_bar_box(combined_summary, "lateral_drift_peak_mean", "Mean Peak Lateral Drift (m)", terrain=None, disturbance_condition=PLOT_DISTURBANCE, plot_kind="box", annotate=True)
    fig.savefig(RESULTS_DIR / "lateral_drift_peak_boxplot.png", dpi=300, bbox_inches="tight")
    plt.show()

    if SAVE_AS_PDF:
        compactify_plot(fig, ax)
        save_paper_fig(fig, RESULTS_DIR / "lateral_drift_peak_boxplot.pdf")

if "lateral_drift_auc_mean" in combined_summary.columns:
    fig, ax = plot_metric_bar_box(combined_summary, "lateral_drift_auc_mean", "Mean Lateral Drift AUC (m)", terrain=PLOT_TERRAIN, disturbance_condition=PLOT_DISTURBANCE, plot_kind="bar", annotate=True)
    fig.savefig(RESULTS_DIR / "lateral_drift_auc_barplot.png", dpi=300, bbox_inches="tight")
    plt.show()

    fig, ax = plot_metric_bar_box(combined_summary, "lateral_drift_auc_mean", "Mean Lateral Drift AUC (m)", terrain=None, disturbance_condition=PLOT_DISTURBANCE, plot_kind="box", annotate=True)
    fig.savefig(RESULTS_DIR / "lateral_drift_auc_boxplot.png", dpi=300, bbox_inches="tight")
    plt.show()

    if SAVE_AS_PDF:
        compactify_plot(fig, ax)
        save_paper_fig(fig, RESULTS_DIR / "lateral_drift_auc_boxplot.pdf")

## Terrain-wise summary views

In [ ]:
# Compact terrain-wise summary for plotting across concrete terrain types.
terrain_rows = combined_summary[combined_summary["terrain"] != "ALL"].copy()
terrain_compact = terrain_rows.groupby(["terrain", "approach_label"], as_index=False).agg(
    num_episodes=("num_episodes", "sum"),
    success_rate=("success_rate", "mean"),
    failure_rate=("failure_rate", "mean"),
    completion_time_s_mean=("completion_time_s_mean", "mean"),
    heading_error_deg_mean=("heading_error_deg_mean", "mean"),
    lateral_drift_peak_mean=("lateral_drift_peak_mean", "mean"),
)
terrain_compact = _sort_with_preferred_order(terrain_compact)
terrain_compact.to_csv(RESULTS_DIR / "terrain_compact_summary.csv", index=False)

# Compact ALL-terrain summary table, useful for reporting the average across all terrains.
all_terrain_compact = combined_summary[combined_summary["terrain"] == "ALL"].copy()
all_terrain_compact = _sort_with_preferred_order(all_terrain_compact)
all_terrain_compact.to_csv(RESULTS_DIR / "all_terrain_compact_summary.csv", index=False)

terrain_compact.round(4)

In [ ]:
def plot_terrain_metric_lines(terrain_summary, metric_col, ylabel, approach_order=None, highlight=None):
    global APPROACH_ORDER, HIGHLIGHT_APPROACH
    old_order, old_highlight = APPROACH_ORDER, HIGHLIGHT_APPROACH
    if approach_order is not None:
        APPROACH_ORDER = approach_order
    if highlight is not None:
        HIGHLIGHT_APPROACH = highlight
    try:
        fig, ax = plt.subplots(figsize=FIGURE_SIZE)
        terrains = [t for t in TERRAINS if t in terrain_summary["terrain"].unique()]
        if not terrains:
            terrains = sorted([t for t in terrain_summary["terrain"].unique() if t != "ALL"])
        labels = _ordered_labels(terrain_summary["approach_label"].dropna().unique())
        markers = ["o", "s", "^", "D", "v", "P", "X", "*"]
        for i, label in enumerate(labels):
            g = terrain_summary[terrain_summary["approach_label"] == label]
            s = g.set_index("terrain")[metric_col]
            y = [s.get(t, np.nan) for t in terrains]
            ax.plot(terrains, y, marker=markers[i % len(markers)], label=label, **_line_style(label))
        ax.set_ylabel(ylabel)
        ax.set_xlabel("Terrain")
        ax.tick_params(axis="x", rotation=30)
        ax.grid(True, axis="y", alpha=0.3)
        ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False, fontsize=8)
        fig.tight_layout()
        return fig, ax
    finally:
        APPROACH_ORDER, HIGHLIGHT_APPROACH = old_order, old_highlight

for metric, ylabel, filename in [
    ("success_rate", "Success Rate", "terrain_success_rate.png"),
    ("completion_time_s_mean", "Completion Time (s)", "terrain_completion_time.png"),
    ("heading_error_deg_mean", "Heading Error (deg)", "terrain_heading_error.png"),
    ("lateral_drift_peak_mean", "Peak Lateral Drift (m)", "terrain_lateral_drift.png"),
]:
    if metric in terrain_compact.columns:
        fig, ax = plot_terrain_metric_lines(terrain_compact, metric, ylabel)
        fig.savefig(RESULTS_DIR / filename, dpi=300, bbox_inches="tight")
        plt.show()

## Selected rollout plotting
The `selected_rollout_metrics.csv` file contains compact per-timestep rows for a few representative episodes per file.

In [ ]:
selected_rollout_metrics[selected_rollout_metrics["approach"] == "go1_pact"].to_csv("pact_terrain_rollout.csv")

In [ ]:
def plot_selected_rollout(rollout_df, approach_label=None, terrain=None, episode=None, metric="heading_error_deg"):
    df = rollout_df.copy()
    if approach_label is not None:
        df = df[df["approach_label"] == approach_label]
    if terrain is not None:
        df = df[df["terrain"] == terrain]
    if episode is not None:
        df = df[df["episode"] == episode]
    if len(df) == 0 or metric not in df.columns:
        print("No rollout rows matched the requested selection or metric is unavailable.")
        return None, None
    first_key = df[["source_file", "episode"]].drop_duplicates().iloc[0]
    df = df[(df["source_file"] == first_key["source_file"]) & (df["episode"] == first_key["episode"])]
    fig, ax = plt.subplots(figsize=(7.0, 3.0))
    ax.plot(df["time_s"], df[metric], linewidth=1.8)
    if "push_change" in df.columns:
        for t in df.loc[df["push_change"] > 0, "time_s"].to_numpy():
            ax.axvline(t, linestyle="--", linewidth=0.8, alpha=0.5)
    ax.set_xlabel("Episode Time (s)")
    ax.set_ylabel(metric)
    ax.grid(True, axis="y", alpha=0.3)
    title = f"{df['approach_label'].iloc[0]} | {df['terrain'].iloc[0]} | episode {df['episode'].iloc[0]}"
    ax.set_title(title)
    fig.tight_layout()
    return fig, ax

# Example:
fig, ax = plot_selected_rollout(selected_rollout_metrics, approach_label="PC", terrain="rough", metric="heading_error_deg")

## Output files

The notebook writes the following compact CSVs:

- `per_file_metrics.csv`: one row per raw CSV with the existing scalar metrics plus aggregate new metrics.
- `per_episode_results.csv`: one row per episode with success/failure, completion time, heading error, and lateral drift summaries.
- `push_event_results.csv`: one row per detected push event with event-triggered lateral drift metrics.
- `selected_rollout_metrics.csv`: compact per-timestep rows for selected representative episodes.
- `combined_summary.csv`: grouped summary table by method, terrain, and disturbance condition.
- `terrain_compact_summary.csv`: terrain-wise method comparison table.